# Notebook 01: sklearn API and Mental Model

## Why This Matters

scikit-learn has become the lingua franca of applied ML for a reason: its consistent estimator API means any classifier, regressor, or transformer works identically within Pipelines, GridSearchCV, and cross-validation. Mastering the mental model lets you write flexible, reusable, leakage-free ML code. The sklearn API appears in nearly every ML interview coding exercise, and subtle misuse (fit_transform on the full dataset) is one of the most common sources of overly optimistic evaluation.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
import warnings; warnings.filterwarnings("ignore")
print("Ready.")

## 1. The Estimator API: fit, predict, transform

Every sklearn object is an estimator. The API has three core methods:

- **`fit(X, y=None)`**: Learn from data. Stores learned parameters as attributes ending in `_`. Returns `self`.
- **`predict(X)`**: Make predictions using learned parameters.
- **`transform(X)`**: Apply a learned transformation to data. Only for transformers.
- **`fit_transform(X, y=None)`**: `fit` then `transform` in one step.

**The data contract**:
- `X` must be array-like of shape `(n_samples, n_features)` - each row is a sample
- `y` must be array-like of shape `(n_samples,)` - one label per sample
- After `fit`, all learned attributes end in `_` (by convention)

**Why consistent API matters**: You can swap any classifier into a Pipeline or GridSearchCV without changing surrounding code.

In [ ]:
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

scaler = StandardScaler()
print("Before fit:", hasattr(scaler, "mean_"))
scaler.fit(X)
print("After fit: ", hasattr(scaler, "mean_"))
print(f"Learned mean (first 5): {scaler.mean_[:5].round(3)}")

X_scaled = scaler.transform(X)
print(f"Scaled mean: {X_scaled.mean():.6f} (should be ~0)")
print(f"Scaled std:  {X_scaled.std():.6f} (should be ~1)")

clf = LogisticRegression(max_iter=200)
clf.fit(X_scaled, y)
print(f"\nLogReg coef shape: {clf.coef_.shape}")
preds = clf.predict(X_scaled)
probs = clf.predict_proba(X_scaled)
print(f"Training accuracy: {(preds == y).mean():.4f}")

## 2. Data Leakage via fit_transform

**The most common sklearn mistake**: calling `fit_transform` on the full dataset before splitting.

```python
# WRONG: leaks test statistics into training
X_scaled = scaler.fit_transform(X)  # scaler sees entire dataset
X_train, X_test = train_test_split(X_scaled, ...)  # too late

# CORRECT: fit only on train, transform both
X_train, X_test = train_test_split(X, ...)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

**Why it matters**: The model implicitly has access to test set statistics during training. For StandardScaler the effect is subtle. For target encoding, it's catastrophic - it directly leaks the label signal.

**Solution**: Use Pipelines. They guarantee fit is called only on training data during cross-validation.

In [ ]:
X, y = make_classification(n_samples=2000, n_features=20, random_state=42)

# WRONG: fit_transform on all data before split
scaler_leaky = StandardScaler()
X_all = scaler_leaky.fit_transform(X)  # sees all data
X_tr_l, X_te_l, y_tr, y_te = train_test_split(X_all, y, test_size=0.3, random_state=42)
clf_l = LogisticRegression(max_iter=200).fit(X_tr_l, y_tr)
acc_leaky = (clf_l.predict(X_te_l) == y_te).mean()

# CORRECT: Pipeline
pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=200))])
X_tr, X_te, y_tr2, y_te2 = train_test_split(X, y, test_size=0.3, random_state=42)
pipe.fit(X_tr, y_tr2)
acc_correct = (pipe.predict(X_te) == y_te2).mean()

print("=== Leakage Demo ===")
print(f"With leakage:    {acc_leaky:.4f}")
print(f"Without leakage: {acc_correct:.4f}")
print("(For StandardScaler the difference is small; for target encoding it is huge)")
cv = cross_val_score(pipe, X, y, cv=5)
print(f"\n5-fold CV: {cv.mean():.4f} +/- {cv.std():.4f}")
print("Pipeline ensures scaler is re-fit on each training fold only.")

## 3. Building a Custom Estimator

Any Python class that implements the right interface can participate in sklearn's ecosystem:
1. Inherit from `BaseEstimator` - provides `get_params` / `set_params` for GridSearchCV
2. For transformers: also inherit from `TransformerMixin` - provides `fit_transform` for free
3. Implement `fit(X, y=None)` - store learned params as `attribute_`
4. Implement `transform(X)` or `predict(X)`
5. No `__init__` side effects - parameters set in `__init__`, computation in `fit`

In [ ]:
class ClipTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, p_low=1, p_high=99):
        self.p_low = p_low; self.p_high = p_high
    
    def fit(self, X, y=None):
        X = np.array(X)
        self.lower_ = np.percentile(X, self.p_low, axis=0)
        self.upper_ = np.percentile(X, self.p_high, axis=0)
        self.n_features_in_ = X.shape[1]
        return self
    
    def transform(self, X):
        return np.clip(np.array(X, dtype=float), self.lower_, self.upper_)

X_demo = np.random.randn(100, 3)
X_demo[0, 0] = 100  # extreme outlier
clipper = ClipTransformer(p_low=5, p_high=95)
X_clipped = clipper.fit_transform(X_demo)
print(f"Before clipping: max f0 = {X_demo[:,0].max():.2f}")
print(f"After clipping:  max f0 = {X_clipped[:,0].max():.2f}")

custom_pipe = Pipeline([("clip", ClipTransformer()), ("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=200))])
X2, y2 = make_classification(n_samples=500, n_features=10, random_state=42)
cv2 = cross_val_score(custom_pipe, X2, y2, cv=5)
print(f"\nCustom pipeline CV: {cv2.mean():.4f} +/- {cv2.std():.4f}")
print("Custom estimator works seamlessly with GridSearchCV and cross_val_score.")

## 4. clone() vs deepcopy

`clone(estimator)` creates a new estimator with the same hyperparameters but **without fitted state**. Critical for:
- Cross-validation: each fold needs an unfitted estimator
- Ensembles: each base learner starts fresh

`copy.deepcopy(estimator)` copies both parameters AND fitted state - useful if you want to save a fitted model, but dangerous in evaluation loops.

In [ ]:
import copy
from sklearn.base import clone

clf = LogisticRegression(C=0.1, max_iter=200)
X3, y3 = make_classification(n_samples=500, random_state=42)
clf.fit(X3, y3)
print(f"Fitted:     has coef_? {hasattr(clf, "coef_")}")

cloned = clone(clf)
print(f"clone():    has coef_? {hasattr(cloned, "coef_")}, C={cloned.C}")

deepcopied = copy.deepcopy(clf)
print(f"deepcopy(): has coef_? {hasattr(deepcopied, "coef_")}, C={deepcopied.C}")
print("\nclone() resets fitted state - use in CV loops.")
print("deepcopy() preserves fitted state - use to save model snapshots.")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| fit / predict / transform | Universal sklearn interface; `fit` stores learned state as `attr_` |
| Data contract | X is (n_samples, n_features); y is (n_samples,) |
| Data leakage | `fit_transform(all_data)` before split leaks test stats into training |
| Pipeline | Chains transformers; guarantees no leakage in cross-validation |
| Custom estimator | Inherit BaseEstimator + TransformerMixin; learn in fit(), apply in transform() |
| clone() | Creates unfitted copy with same hyperparameters - use in CV loops |

### Common Pitfalls
- Calling `fit_transform` on the full dataset before train/test split
- Fitting inside a custom `transform` method
- Not calling `return self` at the end of `fit`
- Storing mutable state in `__init__` instead of `fit` (breaks `clone()`)
